# XGBoost on GPU + Optuna Bayesian (TPE) Tuning

Binary-classification walkthrough:

1. Generate a small **synthetic tabular** dataset and split train / val / test.
2. Train a **baseline XGBoost** on GPU (`device='cuda'`, `tree_method='hist'`),
   reporting **AUC** and **KS** on validation.
3. Tune hyperparameters with **Optuna's TPE sampler** (Bayesian) maximizing
   validation AUC, with **MedianPruner** to kill weak trials early.
4. Refit on train+val with the best params, evaluate on the held-out test set,
   and inspect feature importances.

Swap the synthetic block in Step 1 with your real `X, y` (e.g. encoder
embeddings from `scripts/extract_embeddings.py` joined with a binary label) —
the rest of the notebook is data-agnostic.

In [ ]:
import numpy as np
import pandas as pd
import xgboost as xgb
import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

SEED = 42
np.random.seed(SEED)

# Probe for a usable CUDA device by training a 1-row model.
def _gpu_ok():
    try:
        xgb.train({'device': 'cuda', 'tree_method': 'hist', 'verbosity': 0},
                  xgb.DMatrix(np.zeros((2, 1)), label=[0, 1]), num_boost_round=1)
        return True
    except Exception:
        return False

DEVICE = 'cuda' if _gpu_ok() else 'cpu'
print(f'xgboost {xgb.__version__} | optuna {optuna.__version__} | device: {DEVICE}')

## 1. Synthetic binary-classification data

`make_classification` with informative + redundant features and mild class
imbalance (positive rate ≈ 20%) — close enough to a realistic risk-scoring
setup to exercise both AUC and KS.

In [ ]:
N, D = 50_000, 40
X, y = make_classification(
    n_samples=N, n_features=D,
    n_informative=12, n_redundant=6, n_repeated=0,
    n_clusters_per_class=3, weights=[0.8, 0.2],
    flip_y=0.02, class_sep=1.0, random_state=SEED,
)
feature_names = [f'f{i:02d}' for i in range(D)]

X_tv, X_test, y_tv, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED)
X_train, X_val, y_train, y_val = train_test_split(
    X_tv, y_tv, test_size=0.2, stratify=y_tv, random_state=SEED)

print(f'train={X_train.shape}  val={X_val.shape}  test={X_test.shape}  '
      f'pos_rate={y.mean():.3f}')

## 2. Metrics — AUC & KS

**KS** (Kolmogorov–Smirnov) is the max gap between the CDFs of predicted
scores for positives vs negatives — a standard rank-based metric in credit /
risk modelling. Range [0, 1]; higher is better.

In [ ]:
def ks_score(y_true, y_score):
    y_true = np.asarray(y_true).astype(bool)
    y_score = np.asarray(y_score)
    order = np.argsort(y_score)
    y_sorted = y_true[order]
    pos_total = y_sorted.sum()
    neg_total = len(y_sorted) - pos_total
    if pos_total == 0 or neg_total == 0:
        return 0.0
    cum_pos = np.cumsum(y_sorted) / pos_total
    cum_neg = np.cumsum(~y_sorted) / neg_total
    return float(np.max(np.abs(cum_pos - cum_neg)))

def report(name, y_true, y_score):
    auc = roc_auc_score(y_true, y_score)
    ks = ks_score(y_true, y_score)
    print(f'{name:>10s}  AUC={auc:.4f}  KS={ks:.4f}')
    return auc, ks

## 3. Baseline XGBoost on GPU

Plain `xgb.train` with early stopping on validation AUC. The two GPU knobs are
`device='cuda'` and `tree_method='hist'` (the modern XGBoost ≥ 2.0 API).

In [ ]:
dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=feature_names)
dval   = xgb.DMatrix(X_val,   label=y_val,   feature_names=feature_names)
dtest  = xgb.DMatrix(X_test,  label=y_test,  feature_names=feature_names)

base_params = {
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'tree_method': 'hist',
    'device': DEVICE,
    'eta': 0.05,
    'max_depth': 6,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'min_child_weight': 1.0,
    'reg_lambda': 1.0,
    'verbosity': 0,
    'seed': SEED,
}

baseline = xgb.train(
    base_params, dtrain,
    num_boost_round=2000,
    evals=[(dtrain, 'train'), (dval, 'val')],
    early_stopping_rounds=50,
    verbose_eval=100,
)
print(f'best_iteration={baseline.best_iteration}')

p_val_base  = baseline.predict(dval,  iteration_range=(0, baseline.best_iteration + 1))
p_test_base = baseline.predict(dtest, iteration_range=(0, baseline.best_iteration + 1))
report('val',  y_val,  p_val_base)
report('test', y_test, p_test_base)

## 4. Optuna Bayesian (TPE) tuning

`TPESampler` is Optuna's Bayesian optimizer (Tree-structured Parzen Estimator).
Each trial samples a config, trains on GPU with early stopping, and reports
**val AUC** back. `MedianPruner` kills trials whose mid-training AUC is below
the running median, so we waste no GPU time on obviously bad configs.

Bump `N_TRIALS` for a real run — 40 is enough to see TPE start to home in.

In [ ]:
N_TRIALS = 40
EARLY_STOP = 50
NUM_BOOST_ROUND = 2000

def objective(trial: optuna.Trial) -> float:
    params = {
        'objective': 'binary:logistic',
        'eval_metric': 'auc',
        'tree_method': 'hist',
        'device': DEVICE,
        'verbosity': 0,
        'seed': SEED,
        'eta':              trial.suggest_float('eta', 1e-3, 0.3, log=True),
        'max_depth':        trial.suggest_int('max_depth', 3, 10),
        'min_child_weight': trial.suggest_float('min_child_weight', 1e-2, 10.0, log=True),
        'subsample':        trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'gamma':            trial.suggest_float('gamma', 1e-8, 5.0, log=True),
        'reg_alpha':        trial.suggest_float('reg_alpha', 1e-8, 5.0, log=True),
        'reg_lambda':       trial.suggest_float('reg_lambda', 1e-8, 5.0, log=True),
    }

    pruning_cb = optuna.integration.XGBoostPruningCallback(trial, 'val-auc')
    booster = xgb.train(
        params, dtrain,
        num_boost_round=NUM_BOOST_ROUND,
        evals=[(dval, 'val')],
        early_stopping_rounds=EARLY_STOP,
        callbacks=[pruning_cb],
        verbose_eval=False,
    )
    trial.set_user_attr('best_iteration', int(booster.best_iteration))
    return float(booster.best_score)

study = optuna.create_study(
    direction='maximize',
    sampler=TPESampler(seed=SEED, multivariate=True, n_startup_trials=10),
    pruner=MedianPruner(n_warmup_steps=20),
)
optuna.logging.set_verbosity(optuna.logging.WARNING)
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

print(f'\nbest val AUC: {study.best_value:.4f}  (trial #{study.best_trial.number})')
print('best params:')
for k, v in study.best_params.items():
    print(f'  {k:>18s}: {v}')

### Tuning diagnostics

Quick look at how the search progressed and which hyperparameters mattered.

In [ ]:
hist = pd.DataFrame([{
    'trial': t.number, 'state': t.state.name, 'val_auc': t.value,
    **t.params,
} for t in study.trials])
print('finished trials:', (hist.state == 'COMPLETE').sum(),
      '| pruned:', (hist.state == 'PRUNED').sum())
print('top 5:')
print(hist[hist.state == 'COMPLETE'].nlargest(5, 'val_auc').to_string(index=False))

try:
    importances = optuna.importance.get_param_importances(study)
    print('\nparam importances (fANOVA):')
    for k, v in importances.items():
        print(f'  {k:>18s}: {v:.3f}')
except Exception as e:
    print('importance unavailable:', e)

## 5. Refit on train+val with best params, evaluate on test

Standard pattern: once tuning is done, retrain on `train ∪ val` for the
best-iteration count discovered during tuning, then score the untouched test
set. Compare against the baseline to confirm tuning actually helped.

In [ ]:
best_params = {
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'tree_method': 'hist',
    'device': DEVICE,
    'verbosity': 0,
    'seed': SEED,
    **study.best_params,
}
best_iter = study.best_trial.user_attrs['best_iteration']
# Small headroom since train+val is larger than train alone.
final_rounds = int(best_iter * 1.1) + 1

dtrainval = xgb.DMatrix(
    np.vstack([X_train, X_val]),
    label=np.concatenate([y_train, y_val]),
    feature_names=feature_names,
)
final = xgb.train(best_params, dtrainval, num_boost_round=final_rounds)

p_test = final.predict(dtest)
print('=== baseline ===')
report('test', y_test, p_test_base)
print('=== tuned ===')
report('test', y_test, p_test)

### Feature importances (top 15)

In [ ]:
imp = final.get_score(importance_type='gain')
imp_df = (pd.DataFrame({'feature': list(imp), 'gain': list(imp.values())})
          .sort_values('gain', ascending=False).head(15).reset_index(drop=True))
print(imp_df.to_string(index=False))

## Next steps

- **Real data**: replace Step 1 with your encoder embeddings + binary label.
  If you have a time dimension, prefer a time-based split over `train_test_split`.
- **Scale tuning**: raise `N_TRIALS` (200+ is common) and persist the study
  via `storage='sqlite:///optuna.db'` so it survives kernel restarts and can
  be resumed across runs.
- **Calibration**: AUC/KS are rank metrics — if you need probabilities, add a
  post-hoc isotonic / Platt calibrator on the validation set.
- **Class imbalance**: if positive rate is much smaller than here, tune
  `scale_pos_weight` (start at `n_neg / n_pos`) and consider PR-AUC alongside
  ROC-AUC.